# EA4 — Transición Energética y Vehículos Eléctricos

**Autor:** Fredy Ramírez  
**Pregunta guía:** ¿De qué manera influyen la temperatura ambiental y la hora del día en la eficiencia de recarga y la autonomía proyectada de las flotas de vehículos eléctricos?

Este cuaderno documenta la ingesta, auditoría, limpieza, análisis y verificación de datos. Los hallazgos se redactarán después de ejecutar el análisis; no se inventarán resultados.

## 1. Configuración e importaciones

Ejecuta las celdas en orden. En el terminal, crea y activa un entorno virtual e instala las dependencias desde `requirements.txt`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import os

DATA_DIR = Path('datos')
DATA_DIR.mkdir(exist_ok=True)
DATASET_URL = 'https://huggingface.co/datasets/ProWiddan/Global-EV-Charging-Station-Telemetry'
print('Origen del dataset:', DATASET_URL)

Origen del dataset: https://huggingface.co/datasets/ProWiddan/Global-EV-Charging-Station-Telemetry


## 2. Ingesta de datos desde Hugging Face

Esta celda intenta localizar archivos de datos CSV/JSON/Parquet del repositorio del dataset. Si el repositorio requiere una configuración distinta, revisaremos el listado de archivos y adaptaremos el cargador.

In [2]:
# Instala huggingface_hub si aún no está disponible: pip install huggingface_hub
from huggingface_hub import list_repo_files, hf_hub_download

dataset_repo = 'ProWiddan/Global-EV-Charging-Station-Telemetry'
files = list_repo_files(dataset_repo, repo_type='dataset')
data_files = [f for f in files if f.lower().endswith(('.csv', '.json', '.parquet'))]
print('Archivos de datos encontrados:')
for f in data_files:
    print('-', f)

if not data_files:
    raise FileNotFoundError('No se encontraron CSV, JSON o Parquet en el repositorio. Revisa el listado de archivos del dataset.')

chosen_file = data_files[0]
local_file = hf_hub_download(repo_id=dataset_repo, filename=chosen_file, repo_type='dataset', local_dir=DATA_DIR)
print('Archivo descargado:', local_file)

/workspaces/analitica/.venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Archivos de datos encontrados:
- ev_charging_telemetry.csv
Archivo descargado: /workspaces/analitica/EA4_Actividad_IA/datos/ev_charging_telemetry.csv


In [3]:
# Carga según extensión
ext = Path(chosen_file).suffix.lower()
if ext == '.csv':
    df = pd.read_csv(local_file)
elif ext == '.json':
    df = pd.read_json(local_file)
elif ext == '.parquet':
    df = pd.read_parquet(local_file)
else:
    raise ValueError(f'Formato no admitido: {ext}')

display(df.head())
print('Dimensiones (filas, columnas):', df.shape)

,timestamp,station_id,city,power_kw,session_duration_mins,connector_type,energy_consumed_kwh
0,2025-08-28 04:20:25.163578,ST-995,Oslo,150.0,118,CCS,72.82
1,2025-08-28 06:20:25.163578,ST-939,Amsterdam,150.0,141,Tesla Supercharger,137.69
2,2025-08-28 07:20:25.163578,ST-292,Oslo,22.0,87,Type2,11.18
3,2025-08-28 07:20:25.163578,ST-247,London,350.0,113,Tesla Supercharger,25.38
4,2025-08-28 10:20:25.163578,ST-453,London,350.0,162,CCS,139.86


Dimensiones (filas, columnas): (5000, 7)


## 3. Auditoría inicial de datos

Antes de limpiar, examinamos nombres de columnas, tipos, faltantes y duplicados. No reemplazaremos valores faltantes por cero salvo que el significado del dato justifique esa decisión.

In [4]:
display(pd.DataFrame({'tipo': df.dtypes.astype(str), 'nulos': df.isnull().sum(), 'nulos_%': (df.isnull().mean()*100).round(2)}))
print('Filas duplicadas:', df.duplicated().sum())
print('Columnas disponibles:', df.columns.tolist())

,tipo,nulos,nulos_%
timestamp,str,0,0.0
station_id,str,0,0.0
city,str,0,0.0
power_kw,float64,0,0.0
session_duration_mins,int64,0,0.0
connector_type,str,0,0.0
energy_consumed_kwh,float64,0,0.0


Filas duplicadas: 0
Columnas disponibles: ['timestamp', 'station_id', 'city', 'power_kw', 'session_duration_mins', 'connector_type', 'energy_consumed_kwh']


## 4. Próximas etapas

- Identificar campos de temperatura, fecha/hora, energía, duración, potencia, vehículo y batería.
- Normalizar unidades y documentar datos faltantes.
- Crear variables derivadas con Pandas y NumPy (solo si existen los campos necesarios).
- Segmentar con `.query()`, agrupar con `groupby()` y verificar cálculos mediante máscaras NumPy.
- Construir visualizaciones y dashboard de Streamlit.
- Redactar hallazgos numéricos y decisiones operativas basados en resultados reales.
- Exportar el resumen de auditoría a TXT.

In [5]:
# Convertir timestamp a formato datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Comprobar el resultado
print("Tipo de dato de timestamp:", df['timestamp'].dtype)
print("Primeras fechas:")
display(df[['timestamp']].head())

# Comprobar si la conversión produjo valores inválidos
print("Fechas inválidas:", df['timestamp'].isna().sum())

Tipo de dato de timestamp: datetime64[us]
Primeras fechas:


,timestamp
0,2025-08-28 04:20:25.163578
1,2025-08-28 06:20:25.163578
2,2025-08-28 07:20:25.163578
3,2025-08-28 07:20:25.163578
4,2025-08-28 10:20:25.163578


Fechas inválidas: 0


In [6]:
# Extraer información temporal
df['hora'] = df['timestamp'].dt.hour
df['dia'] = df['timestamp'].dt.day
df['mes'] = df['timestamp'].dt.month
df['dia_semana'] = df['timestamp'].dt.day_name()

# Mostrar las nuevas columnas
display(df[['timestamp', 'hora', 'dia', 'mes', 'dia_semana']].head(10))

# Comprobar rango temporal del dataset
print("Fecha inicial:", df['timestamp'].min())
print("Fecha final:", df['timestamp'].max())

# Horas disponibles
print("Horas registradas:", sorted(df['hora'].unique()))

,timestamp,hora,dia,mes,dia_semana
0,2025-08-28 04:20:25.163578,4,28,8,Thursday
1,2025-08-28 06:20:25.163578,6,28,8,Thursday
2,2025-08-28 07:20:25.163578,7,28,8,Thursday
3,2025-08-28 07:20:25.163578,7,28,8,Thursday
4,2025-08-28 10:20:25.163578,10,28,8,Thursday
5,2025-08-28 12:20:25.163578,12,28,8,Thursday
6,2025-08-28 14:20:25.163578,14,28,8,Thursday
7,2025-08-28 16:20:25.163578,16,28,8,Thursday
8,2025-08-28 18:20:25.163578,18,28,8,Thursday
9,2025-08-28 20:20:25.163578,20,28,8,Thursday


Fecha inicial: 2025-08-28 04:20:25.163578
Fecha final: 2026-08-28 01:20:25.163578
Horas registradas: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


In [9]:
# Estadísticas descriptivas de las variables numéricas
display(df[['power_kw', 'session_duration_mins', 'energy_consumed_kwh']].describe().round(2))

# Número de ciudades registradas
print("Número de ciudades:", df['city'].nunique())
print("\nCiudades:")
print(df['city'].value_counts())

# Tipos de conectores y frecuencia
print("\nTipos de conectores:")
print(df['connector_type'].value_counts())

# Número de estaciones distintas
print("\nNúmero de estaciones:", df['station_id'].nunique())

,power_kw,session_duration_mins,energy_consumed_kwh
count,5000.00,5000.00,5000.00
mean,139.27,102.86,72.93
std,127.37,55.01,39.13
min,22.00,10.00,4.03
25%,22.00,55.00,39.33
50%,50.00,101.00,73.53
75%,150.00,151.00,106.68
max,350.00,199.00,139.99


Número de ciudades: 7

Ciudades:
city
Dubai        748
Oslo         733
London       722
Stockholm    715
Berlin       697
Tokyo        696
Amsterdam    689
Name: count, dtype: int64

Tipos de conectores:
connector_type
CHAdeMO               1280
Type2                 1254
CCS                   1253
Tesla Supercharger    1213
Name: count, dtype: int64

Número de estaciones: 890


In [10]:
# Calcular potencia media observada durante cada sesión (kW)
df['potencia_media_observada_kw'] = (
    df['energy_consumed_kwh'] / (df['session_duration_mins'] / 60)
)

# Comparar con la potencia nominal de la estación
df['ratio_potencia'] = (
    df['potencia_media_observada_kw'] / df['power_kw']
)

# Mostrar algunos registros
display(
    df[['power_kw', 'session_duration_mins',
        'energy_consumed_kwh', 'potencia_media_observada_kw',
        'ratio_potencia']].head(10).round(2)
)

# Estadísticas de las nuevas variables
display(
    df[['potencia_media_observada_kw', 'ratio_potencia']]
    .describe()
    .round(2)
)

,power_kw,session_duration_mins,energy_consumed_kwh,potencia_media_observada_kw,ratio_potencia
0,150.0,118,72.82,37.03,0.25
1,150.0,141,137.69,58.59,0.39
2,22.0,87,11.18,7.71,0.35
3,350.0,113,25.38,13.48,0.04
4,350.0,162,139.86,51.80,0.15
5,22.0,64,100.14,93.88,4.27
6,50.0,162,59.57,22.06,0.44
7,350.0,19,31.11,98.24,0.28
8,50.0,83,38.65,27.94,0.56
9,22.0,135,93.20,41.42,1.88


,potencia_media_observada_kw,ratio_potencia
count,5000.00,5000.00
mean,71.56,1.41
std,91.90,2.77
min,1.47,0.00
25%,22.85,0.17
50%,42.37,0.51
75%,80.52,1.48
max,813.72,36.99


### Análisis de la potencia media observada y detección de anomalías

A partir de las variables de potencia nominal del cargador (`power_kw`), duración de la sesión (`session_duration_mins`) y energía consumida (`energy_consumed_kwh`), se calculó la potencia media observada de cada sesión, convirtiendo previamente la duración de minutos a horas. También se obtuvo un ratio entre la potencia media calculada y la potencia nominal del cargador para evaluar la coherencia de los registros.

Los resultados muestran una potencia media observada de **71,56 kW**, mientras que la mediana es de **42,37 kW**, lo que indica una distribución asimétrica influida por valores elevados. El ratio de potencia presenta una media de **1,41** y un máximo de **36,99**, lo que evidencia la existencia de registros potencialmente anómalos. En algunos casos, la energía consumida y el tiempo de carga implican potencias medias superiores a la potencia nominal del cargador, algo que requiere revisión.

Estas inconsistencias pueden deberse a errores de registro, datos sintéticos, diferencias en la interpretación de las variables o problemas de calidad del dataset. Por tanto, no se consideran automáticamente valores válidos de funcionamiento real.

En conclusión, este análisis demuestra la importancia de validar la coherencia entre las variables energéticas antes de realizar comparaciones o extraer conclusiones sobre el comportamiento de las estaciones de recarga. Las anomalías identificadas se conservarán inicialmente para su estudio y documentación, evitando eliminarlas sin una justificación metodológica.